In [1]:
pip install wbgapi openpyxl

Note: you may need to restart the kernel to use updated packages.


In [2]:
import wbgapi as wb
import pandas as pd
import numpy as np
import os


In [3]:
os.makedirs("Data",exist_ok=True)
    

In [4]:
## STEP 1 = FETCH RAW DATA FROM WORLD BANK
# INDICATORS dictionary

In [5]:
indicators =  {
    "NY.GDP.PCAP.CD":        "gdp_per_capita",   # GDP per person
    "NY.GDP.MKTP.KD.ZG":     "gdp_growth",       # GDP growth %
    "BX.KLT.DINV.WD.GD.ZS":  "fdi_inflows",      # foreign investment
    "SL.UEM.TOTL.ZS":        "unemployment",     # unemployment %
    "SI.POV.DDAY":           "poverty_headcount",
    "SI.POV.GINI":           "gini_index",       # inequality
    # Health
    "SP.DYN.LE00.IN":        "life_expectancy",
    "SP.DYN.IMRT.IN":        "infant_mortality",
    "SH.MED.BEDS.ZS":        "hospital_beds",
    "SP.POP.TOTL":           "population",
    # Education
    "SE.ADT.LITR.ZS":        "literacy_rate",
    "SE.PRM.ENRR":           "school_enrolment_primary",
    "SE.SEC.ENRR":           "school_enrolment_secondary",
    "SE.XPD.TOTL.GD.ZS":     "education_expenditure",
    # Environment
    "EN.ATM.CO2E.PC":        "co2_per_capita",
    "EG.ELC.ACCS.ZS":        "electricity_access",
    "EG.FEC.RNEW.ZS":        "renewable_energy",
    "AG.LND.FRST.ZS":        "forest_area",
    "SH.H2O.BASW.ZS":        "clean_water_access",
    # Gender
    "SL.TLF.CACT.FE.ZS":     "female_labour",
    "SE.ADT.LITR.FE.ZS":     "female_literacy",
    "SG.GEN.PARL.ZS":        "women_parliament",
    "SH.STA.MMRT":           "maternal_mortality",
    "SE.ENR.PRSC.FM.ZS":     "gender_parity",
    # Digital
    "IT.NET.USER.ZS":        "internet_users",
}
# List of Asian Countries

asian_countries = [
    "IND","PAK","BGD","LKA","NPL","BTN","AFG","MDV",
    "CHN","JPN","KOR","MNG","PRK",
    "IDN","THA","VNM","PHL","MYS","MMR","KHM","LAO","SGP","BRN","TLS",
    "KAZ","UZB","TKM","TJK","KGZ",
    "SAU","ARE","IRN","IRQ","TUR","ISR","JOR","KWT",
    "QAT","BHR","OMN","YEM","SYR","LBN","PSE","ARM","AZE","GEO"
]

subregion_map = {
    "IND":"South Asia","PAK":"South Asia","BGD":"South Asia",
    "LKA":"South Asia","NPL":"South Asia","BTN":"South Asia",
    "AFG":"South Asia","MDV":"South Asia",
    "CHN":"East Asia","JPN":"East Asia","KOR":"East Asia",
    "MNG":"East Asia","PRK":"East Asia",
    "IDN":"Southeast Asia","THA":"Southeast Asia","VNM":"Southeast Asia",
    "PHL":"Southeast Asia","MYS":"Southeast Asia","MMR":"Southeast Asia",
    "KHM":"Southeast Asia","LAO":"Southeast Asia","SGP":"Southeast Asia",
    "BRN":"Southeast Asia","TLS":"Southeast Asia",
    "KAZ":"Central Asia","UZB":"Central Asia","TKM":"Central Asia",
    "TJK":"Central Asia","KGZ":"Central Asia",
    "SAU":"West Asia","ARE":"West Asia","IRN":"West Asia","IRQ":"West Asia",
    "TUR":"West Asia","ISR":"West Asia","JOR":"West Asia","KWT":"West Asia",
    "QAT":"West Asia","BHR":"West Asia","OMN":"West Asia","YEM":"West Asia",
    "SYR":"West Asia","LBN":"West Asia","PSE":"West Asia",
    "ARM":"West Asia","AZE":"West Asia","GEO":"West Asia",
}

print("\nFetching indicators from world bank API...")
frames = []

for code,name in indicators.items():
    try:
        df=wb.data.DataFrame(
            code,
            asian_countries,
            time=range(2000,2024),
            labels = True
        )
        df= df.reset_index()
        df= df.melt(
            id_vars = ['economy','Country'],
            var_name='year',
            value_name = name
        )
        frames.append(df)
        print(f" hello {name}")

    except Exception as e:
        print(f"false {name}:{e}")
from functools import reduce 
raw = reduce(
    lambda a,b:pd.merge(
         a, b,
        on=["economy","Country","year"], # common columns
        how="outer"  # keep all data (even if missing)
    ),
    frames
)
raw["sub_region"] = raw["economy"].map(subregion_map)
raw.to_csv("Data/raw_messy_data.csv", index=False)
print(f"\n  Raw dataset saved → raw_messy_data.csv")
print(f"  Shape: {raw.shape[0]} rows × {raw.shape[1]} columns")
        
        
        
        



Fetching indicators from world bank API...
 hello gdp_per_capita
 hello gdp_growth
 hello fdi_inflows
 hello unemployment
 hello poverty_headcount
 hello gini_index
 hello life_expectancy
 hello infant_mortality
 hello hospital_beds
 hello population
 hello literacy_rate
 hello school_enrolment_primary
 hello school_enrolment_secondary
 hello education_expenditure
false co2_per_capita:APIError: JSON decoding error (https://api.worldbank.org/v2/en/sources/2/series/EN.ATM.CO2E.PC/country/IND;PAK;BGD;LKA;NPL;BTN;AFG;MDV;CHN;JPN;KOR;MNG;PRK;IDN;THA;VNM;PHL;MYS;MMR;KHM;LAO;SGP;BRN;TLS;KAZ;UZB;TKM;TJK;KGZ;SAU;ARE;IRN;IRQ;TUR;ISR;JOR;KWT;QAT;BHR;OMN;YEM;SYR;LBN;PSE;ARM;AZE;GEO/time/YR2000;YR2001;YR2002;YR2003;YR2004;YR2005;YR2006;YR2007;YR2008;YR2009;YR2010;YR2011;YR2012;YR2013;YR2014;YR2015;YR2016;YR2017;YR2018;YR2019;YR2020;YR2021;YR2022;YR2023?per_page=1000&page=1&format=json)
 hello electricity_access
 hello renewable_energy
 hello forest_area
 hello clean_water_access
 hello female_labo

,economy,Country,year,internet_users
0,GEO,Georgia,YR2000,0.484746
1,AZE,Azerbaijan,YR2000,0.147758
2,ARM,Armenia,YR2000,1.300470
3,PSE,West Bank and Gaza,YR2000,1.111310
4,LBN,Lebanon,YR2000,7.952740
